# 06.2 — Estudo A: Notação Obstétrica (G/P/A)

Todos os estudos causais baseados em G/P/A com balanço (SMD),
E-value e análise de sensibilidade.

Estudos:
  A1 — Grande Multípara (G>=5) → Retorno 30d
  A2 — Aborto Anterior (A>=1) → Retorno 30d
  A3 — Grande Multípara → Duração Internação
  A4 — Dose-resposta G → Retorno 30d [NOVO]
  A5 — Aborto Recorrente (A>=2) → Retorno 30d [NOVO]
  A6 — Grande Multípara → Retorno 90d e Óbito [NOVO]

Método: AIPW com 5-fold cross-fitting (duplamente robusto)

In [ ]:
from __future__ import annotations
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "dose_media_lbl_fmt": "Média geral ({pct:.1f}%)",
        "dose_xlabel":        "Número de Gestações (G)",
        "dose_ylabel":        "Retorno 30d (%)",
        "dose_title":         "Dose-Resposta: G → Retorno 30d",
        "da_xlabel":          "ATE (pp)",
        "da_title":           "AIPW por Limiar G",
        "fp_xlabel":          "ATE (pp = pontos percentuais | h = horas)",
        "fp_title":           "Forest Plot — Estudo A: Notação Obstétrica (AIPW 5-fold)",
    },
    "en": {
        "dose_media_lbl_fmt": "Overall mean ({pct:.1f}%)",
        "dose_xlabel":        "Number of Pregnancies (G)",
        "dose_ylabel":        "30-day Return (%)",
        "dose_title":         "Dose-Response: G → 30-day Return",
        "da_xlabel":          "ATE (pp)",
        "da_title":           "AIPW by G Threshold",
        "fp_xlabel":          "ATE (pp = percentage points | h = hours)",
        "fp_title":           "Forest Plot — Study A: Obstetric Notation (AIPW 5-fold)",
    },
}
T = _T[LANG]

ROOT    = Path("..").resolve()
ART_DIR = ROOT / "out" / "artifacts_causal_gestacao"
OUT_DIR = ROOT / "out" / "06.2_estudo_a"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregar Artefatos de 06.1

In [ ]:
with open(ART_DIR / "data_gestacional.pkl", "rb") as f:
    _art = pickle.load(f)

df               = _art["df"]
CONFUNDIDORES_BASE = _art["CONFUNDIDORES_BASE"]
top_tipos        = _art["top_tipos"]

print(f"Dados carregados: {len(df):,} registros")
print(f"Confundidores base: {CONFUNDIDORES_BASE}")

## 2. Funções Auxiliares (standalone)

In [ ]:
def make_preprocessor(X):
    """Preprocessador com MICE para numérico e most_frequent para categórico."""
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    transformers = []
    if num:
        transformers.append((
            "num",
            Pipeline([
                ("imp", IterativeImputer(max_iter=10, random_state=42)),
                ("sc",  StandardScaler()),
            ]),
            num,
        ))
    if cat:
        transformers.append((
            "cat",
            Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat,
        ))
    return ColumnTransformer(transformers, remainder="drop")


def aipw_binario(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW com 5-fold cross-fitting para outcome binário."""
    df_a = df_ana[[T_col, Y_col] + [c for c in confundidores if c in df_ana.columns]].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].isin([0, 1])]
    if len(df_a) < 100:
        return None

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(int)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X = pre.fit_transform(df_a[CONF])

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    sc1 = np.zeros(len(Y))
    sc0 = np.zeros(len(Y))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X[tr_idx], T[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out_tr = np.hstack([X[tr_idx], T[tr_idx].reshape(-1, 1)])
        out_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_out_tr, Y[tr_idx])

        X1_te = np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        X0_te = np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        mu1_te = out_f.predict_proba(X1_te)[:, 1]
        mu0_te = out_f.predict_proba(X0_te)[:, 1]

        sc1[te_idx] = mu1_te + T[te_idx] * (Y[te_idx] - mu1_te) / (ps_te + 1e-9)
        sc0[te_idx] = mu0_te + (1 - T[te_idx]) * (Y[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (sc1 - sc0).mean()
    rng = np.random.default_rng(42)
    boots = [
        (sc1[rng.integers(0, len(Y), len(Y))] - sc0[rng.integers(0, len(Y), len(Y))]).mean()
        for _ in range(n_boot)
    ]
    se = np.std(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pval = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n": len(df_a),
        "n_exp": int(T.sum()),
        "prev_Y": round(Y.mean() * 100, 2),
        "ate_pp": round(ate * 100, 4),
        "ic_lo_pp": round(lo * 100, 4),
        "ic_hi_pp": round(hi * 100, 4),
        "se": round(se, 4),
        "p_valor": round(pval, 4),
    }


def aipw_continuo(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW com 5-fold cross-fitting para outcome contínuo."""
    df_a = df_ana[[T_col, Y_col] + [c for c in confundidores if c in df_ana.columns]].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].between(1, 720)]
    if len(df_a) < 100:
        return None

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(float)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X = pre.fit_transform(df_a[CONF])

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    sc1 = np.zeros(len(Y))
    sc0 = np.zeros(len(Y))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X[tr_idx], T[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out_tr = np.hstack([X[tr_idx], T[tr_idx].reshape(-1, 1)])
        out_f = GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_out_tr, Y[tr_idx])

        X1_te = np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        X0_te = np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        mu1_te = out_f.predict(X1_te)
        mu0_te = out_f.predict(X0_te)

        sc1[te_idx] = mu1_te + T[te_idx] * (Y[te_idx] - mu1_te) / (ps_te + 1e-9)
        sc0[te_idx] = mu0_te + (1 - T[te_idx]) * (Y[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (sc1 - sc0).mean()
    rng = np.random.default_rng(42)
    boots = [
        (sc1[rng.integers(0, len(Y), len(Y))] - sc0[rng.integers(0, len(Y), len(Y))]).mean()
        for _ in range(n_boot)
    ]
    se = np.std(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pval = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n": len(df_a),
        "n_exp": int(T.sum()),
        "Y_media_exp":  round(float(Y[T == 1].mean()), 1),
        "Y_media_ctrl": round(float(Y[T == 0].mean()), 1),
        "ate_h":   round(ate, 2),
        "ic_lo_h": round(lo, 2),
        "ic_hi_h": round(hi, 2),
        "se":      round(se, 4),
        "p_valor": round(pval, 4),
    }


def e_value_risk_diff(ate_pp: float, baseline_risk_pct: float):
    """E-value para diferença de risco (VanderWeele & Ding, 2017)."""
    p0 = baseline_risk_pct / 100
    p1 = p0 + ate_pp / 100
    if p1 <= 0 or p0 <= 0:
        return np.nan, np.nan
    rr = max(p1 / p0, p0 / p1)
    ev = rr + np.sqrt(rr * (rr - 1))
    return round(rr, 3), round(ev, 3)


def smd(df_local, col, T_col="T"):
    """Standardized Mean Difference para avaliação de balanço."""
    g1 = pd.to_numeric(df_local[df_local[T_col] == 1][col], errors="coerce").dropna()
    g0 = pd.to_numeric(df_local[df_local[T_col] == 0][col], errors="coerce").dropna()
    if len(g1) == 0 or len(g0) == 0:
        return np.nan
    pooled = np.sqrt((g1.std() ** 2 + g0.std() ** 2) / 2)
    return abs(g1.mean() - g0.mean()) / pooled if pooled > 0 else 0.0


def _print_binario(nome, r):
    if r is None:
        print("  Amostra insuficiente.")
        return
    print(f"  N análise:      {r['n']:,}")
    print(f"  N expostos:     {r['n_exp']:,} ({r['n_exp']/r['n']*100:.1f}%)")
    print(f"  Prevalência Y:  {r['prev_Y']}%")
    print(f"  ATE (AIPW):    {r['ate_pp']:+.2f} pp")
    print(f"  IC95%:         [{r['ic_lo_pp']:+.2f}, {r['ic_hi_pp']:+.2f}]")
    print(f"  p-valor:       {r['p_valor']:.4f}")
    rr, ev = e_value_risk_diff(r["ate_pp"], r["prev_Y"])
    print(f"  E-value:       RR_obs={rr}  E={ev}")


def _print_continuo(nome, r):
    if r is None:
        print("  Amostra insuficiente.")
        return
    print(f"  Duração — expostos: {r['Y_media_exp']:.1f}h  vs  controles: {r['Y_media_ctrl']:.1f}h")
    print(f"  ATE (AIPW):    {r['ate_h']:+.1f}h")
    print(f"  IC95%:         [{r['ic_lo_h']:+.1f}, {r['ic_hi_h']:+.1f}]")
    print(f"  p-valor:       {r['p_valor']:.4f}")


def _smd_report(df_local, T_col, conf_list):
    smds = {}
    for c in conf_list:
        if c in df_local.columns:
            smds[c] = smd(df_local.rename(columns={T_col: "T"}), c)
    unbal = {k: v for k, v in smds.items() if not np.isnan(v) and v > 0.1}
    print(f"  SMD > 0.1 (desbalanceados): {unbal if unbal else 'nenhum'}")
    return smds

## 3. Estudo A1 — Grande Multípara (G>=5) → Retorno 30d

In [ ]:
print("=" * 60)
print("ESTUDO A1 — Grande Multípara (G>=5) → Retorno 30d")
print("=" * 60)

df_gm = df[df["G_text"].notna()].copy()
df_gm["T_grande_multipara"] = (df_gm["G_text"] >= 5).astype(int)
conf_gm = [c for c in CONFUNDIDORES_BASE if c in df_gm.columns]

_smd_report(df_gm.rename(columns={"T_grande_multipara": "T"}), "T", conf_gm)

r_a1 = aipw_binario(df_gm, "T_grande_multipara", "Y_ret30", conf_gm)
_print_binario("A1", r_a1)

## 4. Estudo A2 — Aborto Anterior (A>=1) → Retorno 30d

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A2 — Aborto Anterior (A>=1) → Retorno 30d")
print("=" * 60)

df_ab = df[df["A_text"].notna()].copy()
df_ab["T_aborto_ant"] = (df_ab["A_text"] >= 1).astype(int)
conf_ab = [c for c in CONFUNDIDORES_BASE if c in df_ab.columns]

_smd_report(df_ab.rename(columns={"T_aborto_ant": "T"}), "T", conf_ab)

r_a2 = aipw_binario(df_ab, "T_aborto_ant", "Y_ret30", conf_ab)
_print_binario("A2", r_a2)

## 5. Estudo A3 — Grande Multípara → Duração Internação

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A3 — Grande Multípara → Duração Internação (horas)")
print("=" * 60)

df_dur_gm = df[df["G_text"].notna() & df["Y_dur"].notna()].copy()
df_dur_gm["T_grande_multipara"] = (df_dur_gm["G_text"] >= 5).astype(int)
conf_dur = [c for c in CONFUNDIDORES_BASE if c in df_dur_gm.columns]

r_a3 = aipw_continuo(df_dur_gm, "T_grande_multipara", "Y_dur", conf_dur)
_print_continuo("A3", r_a3)

## 6. Estudo A4 — Dose-Resposta G → Retorno 30d (NOVO)

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A4 — Dose-Resposta G → Retorno 30d")
print("=" * 60)

df_dose = df[df["G_text"].notna() & df["G_text"].between(1, 7)].copy()
dose_tbl = (
    df_dose.groupby("G_text")["Y_ret30"]
    .agg(n="count", n_ret="sum", taxa=lambda x: x.mean() * 100)
    .reset_index()
    .rename(columns={"G_text": "G"})
)
print("\nTaxa de retorno 30d por G:")
print(dose_tbl.to_string(index=False))
dose_tbl.to_csv(OUT_DIR / "dose_resposta_G.csv", index=False)

# AIPW por limiar G
dose_aipw = []
for limiar, label in [(2, "G>=2_vs_G1"), (3, "G>=3_vs_G<=2"), (4, "G>=4_vs_G<=3"), (5, "G>=5_vs_G<=4")]:
    df_tmp = df[df["G_text"].notna()].copy()
    if limiar == 2:
        df_tmp["T_dose"] = (df_tmp["G_text"] >= 2).astype(int)
    else:
        df_tmp["T_dose"] = (df_tmp["G_text"] >= limiar).astype(int)
    conf_tmp = [c for c in CONFUNDIDORES_BASE if c in df_tmp.columns]
    r_tmp = aipw_binario(df_tmp, "T_dose", "Y_ret30", conf_tmp)
    if r_tmp:
        r_tmp["label"] = label
        dose_aipw.append(r_tmp)
        print(f"  {label}: ATE={r_tmp['ate_pp']:+.2f}pp  IC95%[{r_tmp['ic_lo_pp']:+.2f},{r_tmp['ic_hi_pp']:+.2f}]  p={r_tmp['p_valor']:.4f}")

# Line chart dose-resposta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(dose_tbl["G"].astype(str), dose_tbl["taxa"], marker="o", color="#1f77b4", linewidth=2)
ax1.axhline(df["Y_ret30"].mean() * 100, color="black", ls="--", lw=1,
            label=T["dose_media_lbl_fmt"].format(pct=df['Y_ret30'].mean()*100))
for _, row in dose_tbl.iterrows():
    ax1.annotate(f"n={int(row['n'])}", (str(row["G"]), row["taxa"]),
                 textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax1.set_xlabel(T["dose_xlabel"])
ax1.set_ylabel(T["dose_ylabel"])
ax1.set_title(T["dose_title"])
ax1.legend()
ax1.grid(alpha=0.3)

if dose_aipw:
    df_da = pd.DataFrame(dose_aipw)
    ax2 = axes[1]
    y_pos = range(len(df_da))
    for i, row in df_da.iterrows():
        color = "#d62728" if row["ate_pp"] > 0 else "#2ca02c"
        ax2.errorbar(
            row["ate_pp"], i,
            xerr=[[row["ate_pp"] - row["ic_lo_pp"]], [row["ic_hi_pp"] - row["ate_pp"]]],
            fmt="o", color=color, capsize=5, markersize=7,
        )
        sig = "***" if row["p_valor"] < 0.001 else "**" if row["p_valor"] < 0.01 else "*" if row["p_valor"] < 0.05 else "ns"
        ax2.text(row["ic_hi_pp"] + 0.2, i, f"p={row['p_valor']:.3f} {sig}", va="center", fontsize=8)
    ax2.axvline(0, color="black", lw=0.8, ls="--")
    ax2.set_yticks(list(y_pos))
    ax2.set_yticklabels(df_da["label"], fontsize=9)
    ax2.set_xlabel(T["da_xlabel"])
    ax2.set_title(T["da_title"])
    ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "dose_resposta_G.png", dpi=150)
plt.show()
plt.close()

r_a4 = dose_aipw[-1] if dose_aipw else None  # G>=5 vs G<=4

## 7. Estudo A5 — Aborto Recorrente (A>=2) → Retorno 30d (NOVO)

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A5 — Aborto Recorrente (A>=2) → Retorno 30d")
print("=" * 60)

df_ab2 = df[df["A_text"].notna()].copy()
df_ab2["T_aborto_rec"] = (df_ab2["A_text"] >= 2).astype(int)
conf_ab2 = [c for c in CONFUNDIDORES_BASE if c in df_ab2.columns]

_smd_report(df_ab2.rename(columns={"T_aborto_rec": "T"}), "T", conf_ab2)

r_a5 = aipw_binario(df_ab2, "T_aborto_rec", "Y_ret30", conf_ab2)
_print_binario("A5", r_a5)

## 8. Estudo A6 — Grande Multípara → Retorno 90d e Óbito (NOVO)

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A6 — Grande Multípara → Multi-target (Ret90d, Óbito)")
print("=" * 60)

df_gm2 = df[df["G_text"].notna()].copy()
df_gm2["T_grande_multipara"] = (df_gm2["G_text"] >= 5).astype(int)
conf_gm2 = [c for c in CONFUNDIDORES_BASE if c in df_gm2.columns]

r_a6_ret90 = None
r_a6_obito = None

if "Y_ret90" in df_gm2.columns and df_gm2["Y_ret90"].notna().sum() > 100:
    print("\n  A6a — Grande Multípara → Retorno 90d")
    r_a6_ret90 = aipw_binario(df_gm2, "T_grande_multipara", "Y_ret90", conf_gm2)
    _print_binario("A6a", r_a6_ret90)
else:
    print("  Y_ret90 não disponível ou insuficiente.")

print("\n  A6b — Grande Multípara → Óbito")
r_a6_obito = aipw_binario(df_gm2, "T_grande_multipara", "Y_obito", conf_gm2)
_print_binario("A6b", r_a6_obito)

## 9. Análise de Sensibilidade — Threshold G (A1)

In [ ]:
print("\n" + "=" * 60)
print("SENSIBILIDADE — Threshold G para Grande Multípara")
print("=" * 60)

sens_rows = []
for th in [3, 4, 5, 6]:
    df_s = df[df["G_text"].notna()].copy()
    df_s["T_gm"] = (df_s["G_text"] >= th).astype(int)
    if df_s["T_gm"].sum() < 30:
        print(f"  G>={th}: expostos insuficientes ({df_s['T_gm'].sum()})")
        continue
    conf_s = [c for c in CONFUNDIDORES_BASE if c in df_s.columns]
    r_s = aipw_binario(df_s, "T_gm", "Y_ret30", conf_s)
    if r_s:
        r_s["threshold"] = th
        r_s["label"] = f"G>={th}"
        sens_rows.append(r_s)
        rr, ev = e_value_risk_diff(r_s["ate_pp"], r_s["prev_Y"])
        print(f"  G>={th}: n={r_s['n']:,}  n_exp={r_s['n_exp']:,}  "
              f"ATE={r_s['ate_pp']:+.2f}pp  p={r_s['p_valor']:.4f}  E-value={ev}")

df_sens = pd.DataFrame(sens_rows)
df_sens.to_csv(OUT_DIR / "sensibilidade_threshold_G.csv", index=False)
print(f"  Salvo: {OUT_DIR / 'sensibilidade_threshold_G.csv'}")

## 10. Forest Plot — Estudo A

In [ ]:
# Coleta todos os resultados com label unificado
estudos_a = []
for label, r, unit in [
    ("A1 — Grande Multípara → Ret30d",    r_a1, "pp"),
    ("A2 — Aborto Anterior → Ret30d",      r_a2, "pp"),
    ("A3 — Grande Multípara → Duração",    r_a3, "h"),
    ("A4 — G>=5 vs G<=4 → Ret30d",         r_a4, "pp"),
    ("A5 — Aborto Recorrente → Ret30d",    r_a5, "pp"),
    ("A6a — G. Mult. → Ret90d",            r_a6_ret90, "pp"),
    ("A6b — G. Mult. → Óbito",             r_a6_obito, "pp"),
]:
    if r is not None:
        estudos_a.append((label, r, unit))

fig, ax = plt.subplots(figsize=(11, max(4, len(estudos_a) * 1.3)))
y_pos = range(len(estudos_a))

for i, (lbl, r, unit) in enumerate(estudos_a):
    # Corrige o bug 0.0: usa .get() com fallback explícito, não 'or'
    ate = r.get("ate_pp") if r.get("ate_pp") is not None else r.get("ate_h", 0)
    lo  = r.get("ic_lo_pp") if r.get("ic_lo_pp") is not None else r.get("ic_lo_h", 0)
    hi  = r.get("ic_hi_pp") if r.get("ic_hi_pp") is not None else r.get("ic_hi_h", 0)
    cor = "#d62728" if ate > 0 else "#2ca02c"
    ax.errorbar(
        ate, i,
        xerr=[[ate - lo], [hi - ate]],
        fmt="o", color=cor, capsize=5, markersize=7,
    )
    p = r["p_valor"]
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(hi + 0.3, i, f"p={p:.3f} {sig}", va="center", fontsize=8)

ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(list(y_pos))
ax.set_yticklabels([e[0] for e in estudos_a], fontsize=9)
ax.set_xlabel(T["fp_xlabel"])
ax.set_title(T["fp_title"])
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "forest_plot_estudo_A.png", dpi=150)
plt.show()
plt.close()

## 11. Exportar Resultados

In [ ]:
resultados_a = {
    "A1_grande_multipara_ret30d":  r_a1,
    "A2_aborto_anterior_ret30d":   r_a2,
    "A3_grande_multipara_duracao": r_a3,
    "A4_dose_resposta_G5_ret30d":  r_a4,
    "A5_aborto_recorrente_ret30d": r_a5,
    "A6a_grande_mult_ret90d":      r_a6_ret90,
    "A6b_grande_mult_obito":       r_a6_obito,
}

# Salvar em ART_DIR para 06.3 consolidar
(ART_DIR / "resultados_estudo_a.json").write_text(
    json.dumps(resultados_a, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
(OUT_DIR / "resultados_estudo_a.json").write_text(
    json.dumps(resultados_a, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

## Resumo 06.2

In [ ]:
print("\n" + "=" * 60)
print("╔══ 06.2 Estudo A — Notação Obstétrica — CONCLUÍDO ══╗")
print("=" * 60)
for label, r, unit in estudos_a:
    if r is None:
        continue
    ate = r.get("ate_pp") if r.get("ate_pp") is not None else r.get("ate_h", 0)
    lo  = r.get("ic_lo_pp") if r.get("ic_lo_pp") is not None else r.get("ic_lo_h", 0)
    hi  = r.get("ic_hi_pp") if r.get("ic_hi_pp") is not None else r.get("ic_hi_h", 0)
    p   = r["p_valor"]
    sig = "SIGNIFICATIVO" if p < 0.05 else "nao significativo"
    rr, ev = e_value_risk_diff(r.get("ate_pp", 0.0) or 0.0, r.get("prev_Y", 0.0) or 0.0)
    print(f"\n  {label}")
    print(f"  ATE={ate:+.2f} {unit}  IC95%[{lo:+.2f},{hi:+.2f}]  p={p:.4f}  E-value={ev}  -> {sig}")
print("\n" + "=" * 60)
print(f"  Outputs: {OUT_DIR}")
print(f"  JSON compartilhado: {ART_DIR / 'resultados_estudo_a.json'}")
print("=" * 60)